# IELTS Question Generation

**Inputs (source controls):** `PassageContext`, `QuestionType`, `category`, `difficulty`  
**Outputs (targets):** `InstructionContext`, `Question`, `Answer`

What you get:
- Robust loader for **CSV / JSON / JSONL** (`load_table`)
- Schema normalization & filtering (never learns placeholders)
- Special tokens: `<DIFF_EASY|MEDIUM|HARD>`, `<CAT_*>`, `<TYPE_*>`, `<SEP>`, `<BLANK>`
- Grouped split by `passage_id` (leakage-safe)
- Early stopping, label smoothing, warmup; ROUGE-L sanity metric
- Works on older Transformers: `predict_with_generate` set on **Trainer** (not `TrainingArguments`)
- Optional **LoRA (PEFT)** for low VRAM
- Demo generation at the end

> Set `CFG["data_path"]` to your dataset path (e.g., `/kaggle/working/final_data.json`), then run sequentially.

In [1]:
!pip install transformers datasets sentencepiece accelerate peft -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 10.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 41.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 2.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 11.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━

In [2]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=f959f694fa1f732bede537d5a23026857f84eee1a9a6500979a50be4bd93c020
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score


In [3]:
!gdown --id 1heHm5FaSaULfOREzG3mSLt7wAxaO-9wx

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1heHm5FaSaULfOREzG3mSLt7wAxaO-9wx
To: /kaggle/working/final_data.json
100%|██████████████████████████████████████| 15.4M/15.4M [00:00<00:00, 39.9MB/s]


In [4]:
import os, re, hashlib, random, json
from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Tuple
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from torch.utils.data import Dataset
from sklearn.model_selection import GroupShuffleSplit

from transformers import (
    BartForConditionalGeneration,
    BartTokenizerFast,
    T5TokenizerFast,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer, 
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

try:
    from rouge_score import rouge_scorer
except Exception:
    rouge_scorer = None

# PEFT (optional)
HAVE_PEFT = False
try:
    from peft import LoraConfig, get_peft_model, TaskType
    HAVE_PEFT = True
except Exception:
    pass


# Item Response Theory (IRT) Proxy (Scenario 2)
IRT_PROXY_MAP = {
    "Easy": -1.0,
    "Medium": 0.0,
    "Hard": 1.0
}
DEFAULT_B_VALUE = 0.0
# We assume 'a' (discrimination) and 'c' (guessing) are constant for this proxy
DEFAULT_A_VALUE = 1.0
DEFAULT_C_VALUE = 0.0

# Control tokens
DEFAULT_DIFFICULTIES = ["Easy", "Medium", "Hard"]
UTILITY_TOKENS = ["<SEP>", "<BLANK>"]

print("✔️ Imports loaded. PEFT available:", HAVE_PEFT)
print(f"✔️ IRT Proxy Map (Scenario 2) Enabled: {IRT_PROXY_MAP}")

2025-12-03 07:43:04.027825: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764747784.227417      38 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764747784.280693      38 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


✔️ Imports loaded. PEFT available: True
✔️ IRT Proxy Map (Scenario 2) Enabled: {'Easy': -1.0, 'Medium': 0.0, 'Hard': 1.0}


In [19]:
# ==== CONFIG ====
CFG = {
    # Data
    "data_path": "/kaggle/working/final_data.json",
    "output_dir": "./controlable-question-generation-BART",
    
    # Model
    "base_model": "facebook/bart-base",
    
    # Training
    "epochs": 30,
    "batch_size": 4,
    "gradient_accumulation_steps": 4,
    "lr": 8e-5,
    "warmup_ratio": 0.15,
    "weight_decay": 0.01,
    "label_smoothing": 0.1,
    "patience": 7,
    
    # Sequence lengths
    "max_source_len": 512,
    "max_target_len": 256,
    
    # Generation
    "num_beams": 4,
    "length_penalty": 1.0,
    
    # Misc
    "seed": 42,
    "fp16": True,
    "bf16": False,
    "grad_ckpt": False,
    
    # LoRA
    "use_lora": True,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    
    # Curriculum Learning
    "use_curriculum": True,
    "curriculum_stages": [
        {"name": "stage1_easy", "epochs": 5, "filter": ["Easy"], "types": ["readingShortAnswer"]},
        {"name": "stage2_medium", "epochs": 5, "filter": ["Easy", "Medium"], "types": ["readingShortAnswer", "readingTrueFalseNotGiven"]},
        {"name": "stage3_all", "epochs": 15, "filter": None, "types": None},
    ],
    
    # Data Augmentation
    "use_augmentation": False,
    "augmentation_factor": 2.0,
}

print("✔️ Config:", CFG)

✔️ Config: {'data_path': '/kaggle/working/final_data.json', 'output_dir': './controlable-question-generation-BART', 'base_model': 'facebook/bart-base', 'epochs': 30, 'batch_size': 4, 'gradient_accumulation_steps': 4, 'lr': 8e-05, 'warmup_ratio': 0.15, 'weight_decay': 0.01, 'label_smoothing': 0.1, 'patience': 7, 'max_source_len': 512, 'max_target_len': 256, 'num_beams': 4, 'length_penalty': 1.0, 'seed': 42, 'fp16': True, 'bf16': False, 'grad_ckpt': False, 'use_lora': True, 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'use_curriculum': True, 'curriculum_stages': [{'name': 'stage1_easy', 'epochs': 5, 'filter': ['Easy'], 'types': ['readingShortAnswer']}, {'name': 'stage2_medium', 'epochs': 5, 'filter': ['Easy', 'Medium'], 'types': ['readingShortAnswer', 'readingTrueFalseNotGiven']}, {'name': 'stage3_all', 'epochs': 15, 'filter': None, 'types': None}], 'use_augmentation': False, 'augmentation_factor': 2.0}


In [20]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def stable_passage_id(row: pd.Series) -> str:
    if "passage_id" in row and pd.notna(row["passage_id"]):
        return str(row["passage_id"])
    txt = str(row["passage"]).strip()
    return hashlib.md5(txt.encode("utf-8")).hexdigest()

def _canon(s: str) -> str:
    s = str(s).strip().upper().replace("-", "_").replace(" ", "_")
    return "".join(ch for ch in s if ch.isalnum() or ch == "_")

def normalize_sep_blank(text: str) -> str:
    if not isinstance(text, str):
        return text
    out = re.sub(r"\\bSEP\\b", "<SEP>", text)
    out = re.sub(r"\\bBLANK\\b", "<BLANK>", out)
    return out

def load_table(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        df = pd.read_csv(path)
    elif ext in (".json", ".jsonl"):
        try:
            df = pd.read_json(path, lines=(ext == ".jsonl"))
        except ValueError:
            with open(path, "r", encoding="utf-8") as f:
                obj = json.load(f)
            if isinstance(obj, dict) and "data" in obj:
                df = pd.DataFrame(obj["data"])
            elif isinstance(obj, list):
                df = pd.DataFrame(obj)
            else:
                raise ValueError("Unsupported JSON structure")
    else:
        raise ValueError(f"Unsupported extension: {ext}")
    return df

def grouped_splits(df: pd.DataFrame, holdout_frac: float = 0.20, seed: int = 42):
    groups = df["passage_id"].values
    gss = GroupShuffleSplit(n_splits=1, test_size=holdout_frac, random_state=seed)
    train_idx, holdout_idx = next(gss.split(df, groups=groups))
    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_hold = df.iloc[holdout_idx].reset_index(drop=True)
    
    groups_hold = df_hold["passage_id"].values
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
    val_idx, test_idx = next(gss2.split(df_hold, groups=groups_hold))
    df_val = df_hold.iloc[val_idx].reset_index(drop=True)
    df_test = df_hold.iloc[test_idx].reset_index(drop=True)
    
    return df_train, df_val, df_test

# Control tokens
def make_control_tokens_from_df(df: pd.DataFrame) -> Tuple[List[str], List[str]]:
    qts = []
    if "question_type" in df.columns:
        qts = sorted({_canon(t) for t in df["question_type"].dropna().tolist()})
    elif "qtype" in df.columns:
        qts = sorted({_canon(t) for t in df["qtype"].dropna().tolist()})
    
    cat_tokens = []
    type_tokens = [f"<TYPE_{t}>" for t in qts if t]
    return cat_tokens, type_tokens

def as_type_token(qtype: Optional[str]) -> str:
    if not isinstance(qtype, str) or not qtype.strip():
        return ""
    return f"<TYPE_{_canon(qtype)}>"

print("✅ Helper functions defined")

✅ Helper functions defined


In [21]:
PREFIX_MAP = {
    'READINGTRUEFALSENOTGIVEN': 'Generate a TRUE/FALSE/NOT GIVEN statement based on the passage.',
    'READINGYESNONOTGIVEN': 'Generate a YES/NO/NOT GIVEN statement that reflects the writer\'s opinion.',
    'READINGMULTIPLECHOICES': 'Create a multiple-choice question with 4 options (A, B, C, D).',
    'READINGSHORTANSWER': 'Create a short answer question that requires ONE WORD or A NUMBER from the passage.',
    'READINGMATCHINGHEADINGS': 'Create a heading that summarizes a paragraph.',
    'READINGMATCHINGFEATURES': 'Create a statement that can be matched with a person/theory/place mentioned in the passage.',
    'READINGMATCHINGENDINGS': 'Create a sentence opening that requires completion from the passage.',
    'READINGTEXTCOMPLETION': 'Create a summary with blanks to be filled from the passage.',
    'READINGSELECTIVETEXTCOMPLETION': 'Create a summary with multiple choice options for each blank.',
    'READINGTABLECOMPLETION': 'Create a table or form with missing information to be completed.',
}

def get_prefix_for_type(qtype: Optional[str]) -> str:
    if not qtype:
        return "Generate an IELTS reading question."
    canon_type = _canon(qtype)
    return PREFIX_MAP.get(canon_type, "Generate an IELTS reading question.")

print("✅ Prefix map defined")

✅ Prefix map defined


In [8]:
@dataclass
class QGExample:
    source: str
    target: str
    meta: Dict[str, Any]

class QGDataset(Dataset):
    """
    IMPROVED VERSION:
    - Source: [IRT_PARAMS] + Prefix + Type + Passage
    - Target: Instruction\nQuestion\nAnswer (بدون control tokens)
    """
    
    def __init__(
        self, 
        df, 
        tokenizer, 
        max_source_len, 
        max_target_len, 
        default_diff="Medium",
        use_prefix=True
    ):
        self.items: List[QGExample] = []
        self.tok = tokenizer
        self.max_source_len = max_source_len
        self.max_target_len = max_target_len
        self.default_diff = default_diff
        self.use_prefix = use_prefix
        self._build(df)
    
    def _fmt_source(self, passage, difficulty, qtype):
        """
        فرمت جدید (IRT Proxy):
        [a=1.0, b=-1.0, c=0.0] [PREFIX] [TYPE_TOKEN] Passage: {passage}
        """
        # 1. Get the numerical 'b' value from the proxy map
        b_value = IRT_PROXY_MAP.get(difficulty, DEFAULT_B_VALUE)
        
        # 2. Format the IRT parameter string
        irt_string = f"[a={DEFAULT_A_VALUE}, b={b_value}, c={DEFAULT_C_VALUE}]"
        
        # 3. Get other controls
        type_token = as_type_token(qtype)
        prefix = ""
        if self.use_prefix:
            prefix = get_prefix_for_type(qtype)
        
        # 4. Construct the final source string
        return f"{irt_string} {prefix} {type_token}\nPassage: {passage}"
    
    def _fmt_target(self, instruction, question, answer):
        """
        فرمت جدید (ساده‌تر):
        Instruction: {ins}
        Question: {q}
        Answer: {a}
        
        بدون control tokens!
        """
        parts = []
        
        if instruction and instruction.strip():
            parts.append(f"Instruction: {instruction.strip()}")
        
        if question and question.strip():
            parts.append(f"Question: {question.strip()}")
        
        if answer and answer.strip():
            parts.append(f"Answer: {answer.strip()}")
        
        if not parts:
            raise ValueError("Empty target")
        
        return "\n".join(parts)
    
    def _build(self, df):
        bad = 0
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Building dataset"):
            passage = normalize_sep_blank(str(row["passage"]).strip())
            difficulty = str(row.get("difficulty", self.default_diff)).strip().title()
            qtype = row.get("question_type") or row.get("qtype")
            
            instruction = normalize_sep_blank(str(row.get("ref_instruction", "")).strip())
            question = normalize_sep_blank(str(row.get("ref_question", "")).strip())
            answer = normalize_sep_blank(str(row.get("ref_answer", "")).strip())
            
            # Validation
            if not instruction or not question:
                bad += 1
                continue
            
            if len(passage) < 50:
                bad += 1
                continue
            
            try:
                source = self._fmt_source(passage, difficulty, qtype)
                target = self._fmt_target(instruction, question, answer)
            except Exception as e:
                bad += 1
                continue
            
            self.items.append(QGExample(
                source=source,
                target=target,
                meta={
                    "difficulty": difficulty,
                    "question_type": qtype,
                    "passage_id": row.get("passage_id", "unknown")
                }
            ))
        
        if bad:
            print(f"⚠️ Skipped {bad} invalid rows")
    
    def __len__(self):
        return len(self.items)
    
    def __getitem__(self, idx):
        ex = self.items[idx]
        
        model_inputs = self.tok(
            ex.source,
            max_length=self.max_source_len,
            truncation=True,
            padding=False
        )
        
        labels = self.tok(
            text_target=ex.target,
            max_length=self.max_target_len,
            truncation=True,
            padding=False
        )
        
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

print("✅ QGDataset ready (IRT Proxy Version)")

✅ QGDataset ready (IRT Proxy Version)


In [9]:
def add_special_tokens(tokenizer, model, extra_tokens=None):
    base = UTILITY_TOKENS 
    if extra_tokens:
        base += list(sorted(set(t for t in extra_tokens if t)))
    
    special_tokens = {"additional_special_tokens": base}
    added = tokenizer.add_special_tokens(special_tokens)
    
    if added > 0:
        model.resize_token_embeddings(len(tokenizer))
        print(f"✅ Added {added} special tokens (total: {len(base)})")
    
    return added

def compute_metrics_builder(tokenizer):
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        
        # Handle tuple output (logits, past_key_values)
        if isinstance(predictions, tuple):
            predictions = predictions[0]
        
        # Handle 3D tensor (batch, seq, vocab)
        if hasattr(predictions, "ndim") and predictions.ndim == 3:
            predictions = predictions.argmax(axis=-1)
        
        # Replace -100 with pad token
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        
        # Decode
        decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        
        metrics = {}
        
        # Average length
        if decoded_preds:
            metrics["pred_len"] = float(np.mean([len(p.split()) for p in decoded_preds]))
        
        # ROUGE-L
        if rouge_scorer and decoded_preds and decoded_labels:
            scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
            scores = [
                scorer.score(label, pred)["rougeL"].fmeasure
                for pred, label in zip(decoded_preds, decoded_labels)
            ]
            if scores:
                metrics["rougeL"] = float(np.mean(scores))
        
        return metrics
    
    return compute_metrics

def maybe_wrap_lora(model, config):
    if not config.get("use_lora", False):
        print("ℹ️ LoRA disabled")
        return model
    
    if not HAVE_PEFT:
        print("⚠️ PEFT not available, continuing without LoRA")
        return model
    
    print("🔧 Applying LoRA adapters for BART...")
    
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=config.get("lora_r", 16),
        lora_alpha=config.get("lora_alpha", 32),
        lora_dropout=config.get("lora_dropout", 0.05),
        bias="none",
        # 👇 CHANGED: specific modules for BART
        target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]
    )
    
    model = get_peft_model(model, lora_config)
    try:
        model.print_trainable_parameters()
    except:
        pass
    
    return model

@torch.no_grad()
def demo_generation(model, tokenizer, test_cases, max_source_len, max_target_len, device):
    """
    تست تولید سوال با control tokens مختلف (ورژن IRT)
    """
    model.eval()
    model.to(device)
    
    print("\n" + "="*80)
    print("🎯 DEMO GENERATION - Testing IRT Proxy Control")
    print("="*80)
    
    for i, (diff, qtype, passage_snippet) in enumerate(test_cases, 1):
        
        # IRT Logic
        b_value = IRT_PROXY_MAP.get(diff, DEFAULT_B_VALUE)
        irt_string = f"[a={DEFAULT_A_VALUE}, b={b_value}, c={DEFAULT_C_VALUE}]"

        type_token = as_type_token(qtype)
        prefix = get_prefix_for_type(qtype)
        
        source = f"{irt_string} {prefix} {type_token}\nPassage: {passage_snippet}"
        
        inputs = tokenizer(
            source,
            return_tensors="pt",
            truncation=True,
            max_length=max_source_len
        ).to(device)
        
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_target_len,
            num_beams=4,
            length_penalty=1.0,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
        
        result = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        print(f"\n--- Test Case {i} ---")
        print(f"Difficulty: {diff} (b={b_value})")
        print(f"Type: {qtype}")
        print(f"\nGenerated Output:")
        print(result)
        print("-" * 80)

print("✅ Training utilities ready (IRT Proxy Version)")

✅ Training utilities ready (IRT Proxy Version)


In [10]:
seed_everything(CFG["seed"])

df = load_table(CFG["data_path"])
print(f"📊 Raw data shape: {df.shape}")
print(f"📊 Columns: {list(df.columns)}")

rename_map = {
    "PassageContext": "passage",
    "QuestionType": "question_type",
    "InstructionContext": "ref_instruction",
    "Question": "ref_question",
    "Answer": "ref_answer",
}
df = df.rename(columns=rename_map)

if "passage_id" not in df.columns:
    df["passage_id"] = df.apply(stable_passage_id, axis=1)

text_cols = ["passage", "ref_instruction", "ref_question", "ref_answer", "difficulty"]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].apply(
            lambda x: normalize_sep_blank(str(x)) if pd.notna(x) else x
        )

before = len(df)
df = df[df["ref_instruction"].fillna("").str.strip().ne("")]
df = df[df["ref_question"].fillna("").str.strip().ne("")]
df = df[df["passage"].fillna("").str.len() >= 50] 
after = len(df)
print(f"✅ Filtered for incomplete rows: {after}/{before} rows remaining")

# Filter question types with distribution < 50
print(f"\nFiltering by question type distribution (threshold >= 50)...")
before_dist_filter = len(df)

type_counts_map = df['question_type'].map(df['question_type'].value_counts())

# Filter the DataFrame to keep only rows where the count is 50 or more
df = df[type_counts_map >= 50].reset_index(drop=True)

after_dist_filter = len(df)
print(f"✅ Filtered by distribution: {after_dist_filter}/{before_dist_filter} rows remaining")

print("\n📊 Dataset Statistics:")
print(f"Difficulty distribution:\n{df['difficulty'].value_counts()}")
print(f"\nQuestion type distribution:\n{df['question_type'].value_counts()}")

cat_tokens, type_tokens = make_control_tokens_from_df(df)
extra_tokens = cat_tokens + type_tokens
print(f"\n🏷️ Control tokens: {len(extra_tokens)}")
print(f"Sample: {extra_tokens[:5]}")

# Split data
train_df, val_df, test_df = grouped_splits(df, holdout_frac=0.20, seed=CFG["seed"])
print(f"\n✅ Split: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

print("\n✅ Data loading complete!")

📊 Raw data shape: (2601, 10)
📊 Columns: ['Topic', 'PassageContext', 'QuestionType', 'InstructionContext', 'Question', 'Answer', 'category', 'difficulty', 'category_confidence', 'difficulty_confidence']
✅ Filtered for incomplete rows: 2596/2601 rows remaining

Filtering by question type distribution (threshold >= 50)...
✅ Filtered by distribution: 2535/2596 rows remaining

📊 Dataset Statistics:
Difficulty distribution:
difficulty
Easy      977
Medium    941
Hard      617
Name: count, dtype: int64

Question type distribution:
question_type
readingTextCompletion       986
readingShortAnswer          537
readingTrueFalseNotGiven    479
readingMultipleChoices      285
readingMatchingFeatures     156
readingYesNoNotGiven         92
Name: count, dtype: int64

🏷️ Control tokens: 6
Sample: ['<TYPE_READINGMATCHINGFEATURES>', '<TYPE_READINGMULTIPLECHOICES>', '<TYPE_READINGSHORTANSWER>', '<TYPE_READINGTEXTCOMPLETION>', '<TYPE_READINGTRUEFALSENOTGIVEN>']

✅ Split: Train=2026, Val=239, Test=270

✅ D

In [12]:
print("🔧 Loading model and tokenizer...")

tokenizer = BartTokenizerFast.from_pretrained(CFG["base_model"])
model = BartForConditionalGeneration.from_pretrained(CFG["base_model"])

add_special_tokens(tokenizer, model, extra_tokens=extra_tokens)

if CFG["grad_ckpt"]:
    model.gradient_checkpointing_enable()
    print("✅ Gradient checkpointing enabled")

model = maybe_wrap_lora(model, CFG)

model.config.max_length = CFG["max_target_len"]
model.config.num_beams = CFG["num_beams"]
model.config.length_penalty = CFG["length_penalty"]
model.config.early_stopping = True

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Model loaded on: {device}")

🔧 Loading model and tokenizer...
✅ Added 8 special tokens (total: 14)
🔧 Applying LoRA adapters for BART...
trainable params: 3,244,032 || all params: 142,670,592 || trainable%: 2.2738
✅ Model loaded on: cuda


In [13]:
train_ds = QGDataset(
    train_df,
    tokenizer,
    CFG["max_source_len"],
    CFG["max_target_len"],
    use_prefix=True
)

val_ds = QGDataset(
    val_df,
    tokenizer,
    CFG["max_source_len"],
    CFG["max_target_len"],
    use_prefix=True
)

print(f"📊 Train: {len(train_ds)}, Val: {len(val_ds)}")

Building dataset: 100%|██████████| 239/239 [00:00<00:00, 10175.08it/s]

📊 Train: 2026, Val: 239


In [14]:
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

In [15]:
train_ds

In [16]:
train_ds[0]

{'input_ids': [0, 10975, 102, 5214, 134, 4, 288, 6, 741, 48771, 134, 4, 288, 6, 740, 5214, 288, 4, 288, 742, 15745, 877, 10, 44929, 73, 597, 49146, 73, 37049, 272, 6372, 2796, 445, 716, 15, 5, 9078, 4, 1437, 50271, 50118, 20340, 1580, 35, 1590, 5, 2958, 8, 3821, 11505, 6, 5, 24696, 9, 5, 2297, 12, 1208, 982, 9, 13401, 8, 3288, 281, 5652, 11, 369, 12, 16507, 666, 2226, 10, 5448, 9, 8079, 899, 7, 2382, 6, 2310, 24956, 148, 5, 3841, 191, 13, 4835, 6, 30260, 6, 30221, 3122, 8, 20500, 4, 635, 6, 5, 11382, 9, 42, 26101, 126, 5, 1149, 3056, 126, 1411, 1684, 63, 43667, 2502, 4, 50118, 50118, 47814, 7, 5, 976, 6, 1149, 3056, 29, 32, 747, 12970, 28108, 2632, 8, 10104, 3924, 11, 1836, 8, 3989, 4, 1590, 49, 17232, 1208, 6, 51, 58, 2127, 9, 5660, 6, 9, 16906, 6, 9, 26545, 8, 9, 13405, 13, 17603, 9, 70, 53, 5, 3912, 2471, 293, 4, 1993, 1149, 3056, 29, 32, 303, 30176, 198, 5, 10348, 911, 9, 13401, 36, 8569, 51, 32, 373, 748, 1469, 4839, 8, 3288, 281, 5652, 36, 8569, 51, 32, 684, 25, 17279, 6249, 3131

In [17]:
training_args = Seq2SeqTrainingArguments(
    output_dir=CFG["output_dir"],
    per_device_train_batch_size=CFG["batch_size"],
    per_device_eval_batch_size=CFG["batch_size"],
    learning_rate=CFG["lr"],
    num_train_epochs=CFG["epochs"],
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=50,
    warmup_ratio=CFG["warmup_ratio"],
    weight_decay=CFG["weight_decay"],
    label_smoothing_factor=CFG["label_smoothing"],
    fp16=CFG["fp16"],
    bf16=CFG["bf16"],
    gradient_accumulation_steps=CFG["gradient_accumulation_steps"],
    report_to=["none"],
    remove_unused_columns=False,
    dataloader_num_workers=2,
    predict_with_generate=True,
    generation_max_length=CFG["max_target_len"],
    generation_num_beams=CFG["num_beams"],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=collator,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics_builder(tokenizer),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CFG["patience"])],
)

/tmp/ipykernel_38/4000477730.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [18]:
train_output = trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Pred Len,Rougel
1,6.119600,5.169287,178.933054,0.153398
2,5.264700,3.869507,56.815900,0.231973
3,4.324500,3.090646,53.146444,0.326441
4,3.226100,2.702709,39.924686,0.419336
5,2.883000,2.452245,44.372385,0.427317
6,2.748500,2.375723,41.728033,0.438187
7,2.626800,2.309494,39.677824,0.452278
8,2.531000,2.267168,43.460251,0.452273
9,2.502800,2.238879,42.962343,0.494647
10,2.407900,2.207757,43.631799,0.499789


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/usr/local/lib/python3.11/dist-packages/transformers/generation/utils.py:1739: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/ge

In [22]:
print("💾 Saving model/tokenizer to:", CFG["output_dir"])
trainer.save_model(CFG["output_dir"])
tokenizer.save_pretrained(CFG["output_dir"])

if len(test_df) > 0:
    print("🧪 Final test evaluation…")
    test_ds = QGDataset(test_df, tokenizer, CFG["max_source_len"], CFG["max_target_len"])
    metrics = trainer.evaluate(eval_dataset=test_ds)
    print(metrics)
else:
    print("No test split available.")

💾 Saving model/tokenizer to: ./controlable-question-generation-BART


/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:252: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


🧪 Final test evaluation…


Building dataset: 100%|██████████| 270/270 [00:00<00:00, 9938.50it/s]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 2.128873109817505, 'eval_pred_len': 43.67037037037037, 'eval_rougeL': 0.5001806675494133, 'eval_runtime': 87.2022, 'eval_samples_per_second': 3.096, 'eval_steps_per_second': 0.39, 'epoch': 30.0}


In [23]:
if len(val_df) > 0:
    sample_df = val_df.sample(10, random_state=CFG["seed"])
    
    test_cases_list = [
        (row["difficulty"], row["question_type"], row["passage"][:1024])
        for _, row in sample_df.iterrows()
    ]
    print(f"--- Generating {len(test_cases_list)} demo samples ---")

    demo_generation(
        trainer.model, 
        tokenizer, 
        test_cases_list, 
        CFG["max_source_len"], 
        CFG["max_target_len"], 
        device
    )
else:
    print("No validation split to sample from.")

--- Generating 10 demo samples ---

🎯 DEMO GENERATION - Testing IRT Proxy Control

--- Test Case 1 ---
Difficulty: Hard (b=1.0)
Type: readingTrueFalseNotGiven

Generated Output:
Instruction: Do the following statement agree with the information in the passage?
Write TRUE, FALSE or NOT GIVEN.
Question: Toward the end of the Middle Ages, the European middle classes began to desire the lifestyle of the elite, including their consumption of spices.
Answer: TRUE
--------------------------------------------------------------------------------

--- Test Case 2 ---
Difficulty: Hard (b=1.0)
Type: readingMatchingFeatures

Generated Output:
Instruction: Reading Passage 2 has eight paragraphs, A-G .
Which paragraph contains the following information?
Write the correct letter, A–G , in boxes 14-19 on your answer sheet.
NB You may use any letter more than once.
Question: a reference to a particular IQ test
Answer: C
--------------------------------------------------------------------------------

--

In [25]:
HUB_MODEL_ID = "alikarimiaca/controlable-question-generation-BART-base"
model.push_to_hub(HUB_MODEL_ID)
tokenizer.push_to_hub(HUB_MODEL_ID)
trainer.push_to_hub(HUB_MODEL_ID)
print("\nAll done! Check your Hugging Face profile.")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


All done! Check your Hugging Face profile.


## **Evaluation**

In [2]:
from transformers import T5ForConditionalGeneration, T5TokenizerFast

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
import torch

BASE_MODEL_NAME = "t5-base"
FINETUNED_MODEL_NAME = "alikarimiaca/controlable-question-generation-T5-base"

tokenizer = AutoTokenizer.from_pretrained(FINETUNED_MODEL_NAME, use_fast=True)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_NAME)
base_model.resize_token_embeddings(len(tokenizer))
model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL_NAME)

model.eval()
print("✅ Fine-Tuned Model loaded successfully!")

✅ Fine-Tuned Model loaded successfully!


In [65]:
print("Tokenizer size:", len(tokenizer))
print("Embeddings:", model.get_input_embeddings().weight.shape)  # should be (len(tokenizer), hidden_size)

Tokenizer size: 32108
Embeddings: torch.Size([32108, 768])


In [26]:
import re 

DEFAULT_DIFFICULTY = "Medium"

def as_cat_token(category: Optional[str]) -> str:
    return "" 

def format_source(passage, difficulty, category=None, qtype=None):
    """Creates the model's input string (IRT Proxy Version)."""
    # IRT Logic
    b_value = IRT_PROXY_MAP.get(difficulty, DEFAULT_B_VALUE)
    irt_string = f"[a={DEFAULT_A_VALUE}, b={b_value}, c={DEFAULT_C_VALUE}]"
    
    type_token = as_type_token(qtype)
    prefix = get_prefix_for_type(qtype)
    return f"{irt_string} {prefix} {type_token}\nPassage: {passage}"


def format_target(difficulty, instruction, question, answer, category=None, qtype=None):
    """
    This function is only for reference/comparison, not for model input.
    Using the simple format.
    """
    ins = (instruction or "").strip()
    q   = (question or "").strip()
    a   = (answer or "").strip()
    if not ins or not q:
        return None 
    
    parts = []
    if ins: parts.append(f"Instruction: {ins}")
    if q: parts.append(f"Question: {q}")
    if a: parts.append(f"Answer: {a}")
    return "\n".join(parts)

def parse_generated_output(text: str) -> Dict[str, str]:
    """Parses the model's full output string into components by splitting."""
    
    instruction = ""
    question = ""
    answer = ""
    
    # 1. Split by "Answer:"
    answer_parts = re.split(r"Answer:", text, maxsplit=1, flags=re.IGNORECASE)
    if len(answer_parts) == 2:
        main_part = answer_parts[0]
        answer = answer_parts[1].strip()
    else:
        main_part = answer_parts[0]
        answer = ""
        
    # 2. Split the main part by "Question:"
    question_parts = re.split(r"Question:", main_part, maxsplit=1, flags=re.IGNORECASE)
    if len(question_parts) == 2:
        inst_part = question_parts[0]
        question = question_parts[1].strip()
    else:
        inst_part = question_parts[0]
        question = ""
        
    # 3. Clean up the instruction part (remove "Instruction:")
    instruction = re.sub(r"^\s*Instruction:", "", inst_part, flags=re.IGNORECASE).strip()

    return {
        "instruction": instruction,
        "question": question,
        "answer": answer
    }

print("✔️ Formatting and Parsing functions defined (IRT Proxy Version for Eval).")

✔️ Formatting and Parsing functions defined (IRT Proxy Version for Eval).


In [27]:
results = []

evaluation_set = val_df.head(200) 

print(f"Generating predictions for {len(evaluation_set)} test samples...")

model.to(device)

for _, row in tqdm(evaluation_set.iterrows(), total=len(evaluation_set)):
    
    # Get reference data from the row
    difficulty = str(row.get("difficulty", DEFAULT_DIFFICULTY)).strip().title()
    category   = row.get("category", None)
    qtype      = row.get("question_type", None) if "question_type" in row.index else row.get("qtype", None)
    passage    = normalize_sep_blank(str(row["passage"]).strip())
    
    ref_instruction = normalize_sep_blank(row.get("ref_instruction", ""))
    ref_question    = normalize_sep_blank(row.get("ref_question", ""))
    ref_answer      = normalize_sep_blank(row.get("ref_answer", ""))

    # 1. Format the source input
    source_text = format_source(passage, difficulty, category, qtype)
    
    # 2. Tokenize and generate
    inputs = tokenizer(
        source_text, 
        max_length=CFG["max_source_len"], 
        truncation=True, 
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CFG["max_target_len"],
            num_beams=6,
            do_sample=True,
            temperature=0.15,
            top_p=0.95,
            num_return_sequences=3,
            return_dict_in_generate=True,
            output_scores=True,
            early_stopping=True
        )

    best_sequence_index = torch.argmax(outputs.sequences_scores) 
    best_generated_ids = outputs.sequences[best_sequence_index]
    generated_text = tokenizer.decode(best_generated_ids, skip_special_tokens=True)

    # parse output
    parsed_output = parse_generated_output(generated_text)
    
    # Store all relevant information
    results.append({
        # Reference data
        "passage_id": row["passage_id"],
        "passage": passage,
        "difficulty": difficulty,
        "category": category,
        "question_type": qtype,
        "reference_instruction": ref_instruction,
        "reference_question": ref_question,
        "reference_answer": ref_answer,
        
        # Generated data (parsed)
        "generated_instruction": parsed_output["instruction"],
        "generated_question": parsed_output["question"],
        "generated_answer": parsed_output["answer"],
        
        # Full text for debugging
        "generated_full_output": generated_text
    })

print("Generation complete.")
eval_df = pd.DataFrame(results)

Generating predictions for 200 test samples...


100%|██████████| 200/200 [04:09<00:00,  1.25s/it]

Generation complete.


In [28]:
display_df = eval_df.rename(columns={
    "passage": "reference_passage",
    "difficulty": "reference_difficulty",
    "category": "reference_category",
    "question_type": "reference_question_type"
})

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 300)

print("Displaying Evaluation Results:")

display(display_df[[
    "reference_passage",
    "reference_difficulty", 
    "reference_category", 
    "reference_question_type",
    "reference_instruction", 
    "reference_question", 
    "reference_answer",
    
    "generated_full_output",
    "generated_instruction", 
    "generated_question", 
    "generated_answer",
]])

Displaying Evaluation Results:


,reference_passage,reference_difficulty,reference_category,reference_question_type,reference_instruction,reference_question,reference_answer,generated_full_output,generated_instruction,generated_question,generated_answer
0,"Internationally, ‘giftedness’ is most frequently determined by a score on a general intelligence test, known as an IQ test, which is above a chosen cutoff point, usually at around the top 2-5%. Children’s educational environment contributes to the IQ score and the way intelligence is used. For e...",Hard,Education/Psychology,readingMatchingFeatures,"Reading Passage 2 has six paragraphs, A-F .\nWhich paragraph contains the following information?\nWrite the correct letter, A-F , in boxes 14-17 on your answer sheet\nNB You may use any letter more than once.",a reference to the influence of the domestic background on the gifted child.,A,"Instruction: Reading Passage 2 has eight paragraphs, A-G .\nWhich paragraph contains the following information?\nWrite the correct letter, A–G , in boxes 14-19 on your answer sheet.\nNB You may use any letter more than once.\nList of Researchers\nA Freeman, 2010\nBFreeman,2010\nC Freeman\nDFreem...","Reading Passage 2 has eight paragraphs, A-G .\nWhich paragraph contains the following information?\nWrite the correct letter, A–G , in boxes 14-19 on your answer sheet.\nNB You may use any letter more than once.\nList of Researchers\nA Freeman, 2010\nBFreeman,2010\nC Freeman\nDFreeman","The vocabulary aspect, for example, is dependent on having heard those words.",C
1,"Internationally, ‘giftedness’ is most frequently determined by a score on a general intelligence test, known as an IQ test, which is above a chosen cutoff point, usually at around the top 2-5%. Children’s educational environment contributes to the IQ score and the way intelligence is used. For e...",Hard,Education/Psychology,readingMatchingFeatures,"Reading Passage 2 has six paragraphs, A-F .\nWhich paragraph contains the following information?\nWrite the correct letter, A-F , in boxes 14-17 on your answer sheet\nNB You may use any letter more than once.",reference to what can be lost if learners are given too much guidance.,D,"Instruction: Reading Passage 2 has eight paragraphs, A-G .\nWhich paragraph contains the following information?\nWrite the correct letter, A–G , in boxes 14-19 on your answer sheet.\nNB You may use any letter more than once.\nList of Researchers\nA Freeman, 2010\nBFreeman,2010\nC Freeman\nDFreem...","Reading Passage 2 has eight paragraphs, A-G .\nWhich paragraph contains the following information?\nWrite the correct letter, A–G , in boxes 14-19 on your answer sheet.\nNB You may use any letter more than once.\nList of Researchers\nA Freeman, 2010\nBFreeman,2010\nC Freeman\nDFreeman","The vocabulary aspect, for example, is dependent on having heard those words.",C
2,"Internationally, ‘giftedness’ is most frequently determined by a score on a general intelligence test, known as an IQ test, which is above a chosen cutoff point, usually at around the top 2-5%. Children’s educational environment contributes to the IQ score and the way intelligence is used. For e...",Hard,Education/Psychology,readingMatchingFeatures,"Reading Passage 2 has six paragraphs, A-F .\nWhich paragraph contains the following information?\nWrite the correct letter, A-F , in boxes 14-17 on your answer sheet\nNB You may use any letter more than once.",a reference to the damaging effects of anxiety.,F,"Instruction: Reading Passage 2 has eight paragraphs, A-G .\nWhich paragraph contains the following information?\nWrite the correct letter, A–G , in boxes 14-19 on your answer sheet.\nNB You may use any letter more than once.\nList of Researchers\nA Freeman, 2010\nBFreeman,2010\nC Freeman\nDFreem...","Reading Passage 2 has eight paragraphs, A-G .\nWhich paragraph contains the following information?\nWrite the correct letter, A–G , in boxes 14-19 on your answer sheet.\nNB You may use any letter more than once.\nList of Researchers\nA

In [29]:
display_df.iloc[3]['generated_full_output']

'Instruction: Reading Passage 2 has eight paragraphs, A-G .\nWhich paragraph contains the following information?\nWrite the correct letter, A–G , in boxes 14-19 on your answer sheet.\nNB You may use any letter more than once.\nList of Researchers\nA Freeman, 2010\nBFreeman,2010\nC Freeman\nDFreeman\nQuestion: The vocabulary aspect, for example, is dependent on having heard those words.\nAnswer: C'

In [30]:
output_filename = "evaluation_results_BART.csv"

display_df.to_csv(output_filename, index=False)

print(f"Successfully saved results to {output_filename}")

Successfully saved results to evaluation_results_BART.csv
